# Week 13 Random Forest and Extra Trees - Lecture
When the Decision Tree model is not productive or makes good predictions, you can use the Random Forest or Extra Trees model to improve predictions.<br>

The data set you will use to create a Random Forest Model is the Heart Data set. The columns you will use are listed below.
Feature	Description	Example Values
- age	Age in years	29, 45, 60
- sex	1 = male; 0 = female	0, 1
- cp	Chest pain type	0: Typical angina (chest pain), 1: Atypical angina (chest pain not related to heart), 2: Non-anginal pain (typically esophageal spasms (non heart related), 3: Asymptomatic (chest pain not showing signs of disease)
- trestbps	Resting blood pressure (in mm Hg on admission to the hospital)	120, 140, 150
- chol	Serum cholesterol in mg/dl	180, 220, 250
- fbs	Fasting blood sugar > 120 mg/dl (1 = true; 0 = false)	0, 1
- restecg	Resting electrocardiographic results	0: Nothing to note, 1: ST-T Wave abnormality, 2: Left ventricular hypertrophy
- thalach	Maximum heart rate achieved	160, 180, 190
- exang	Exercise induced angina (1 = yes; 0 = no)	0, 1
- oldpeak	ST depression (heart potentially not getting enough oxygen) induced by exercise relative to rest	0.5, 1.0, 2.0
- slope	The slope of the peak exercise ST segment	0: Upsloping, 1: Flatsloping, 2: Downsloping
- ca	Number of major vessels (0-3) colored by fluoroscopy	0, 1, 2, 3
- thal	Thalium stress result	1: Normal, 3: Normal, 6: Fixed defect, 7: Reversible defect
- target	Have disease or not (1 = yes; 0 = no)	0, 1
Note: No personal identifiable information (PPI) can be found in the dataset.



In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
from sklearn.datasets import load_breast_cancer
heart = pd.read_csv("heart.csv")
heart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB


# Make columns that are categorical strings


In [3]:
c_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

for col in c_features:
    heart[col] = heart[col].astype('object')

# Drop columns
Remove columns to drop that have unique to individuals or contain too many missing values to be useful for a general model.  Drop the columns.
- slope
- fbs

In [4]:
LX = ['slope', 'fbs']
heart.drop(columns = LX, axis = 1, inplace = True)

# Display first five records


In [5]:
heart.head()

,age,sex,cp,trestbps,chol,restecg,thalach,exang,oldpeak,ca,thal,target
0,52,1,0,125,212,1,168,0,1.0,2,3,0
1,53,1,0,140,203,0,155,1,3.1,0,3,0
2,70,1,0,145,174,1,125,1,2.6,0,3,0
3,61,1,0,148,203,1,161,0,0.0,1,3,0
4,62,0,0,138,294,1,106,0,1.9,3,2,0


In [6]:
heart['target'].value_counts(normalize = True) * 100

target
1    51.317073
0    48.682927
Name: proportion, dtype: float64

# Split the data
Split the titanic data set into train and test data sets.
- 30% test_size = 0.30, random_state = 42 and 
- stratify = titanic['Survived']

In [7]:
train, test = train_test_split(heart, test_size = 0.30, random_state = 42,
                              stratify = heart['target'])               

# Check distribution
- Make sure the distribution of Survived is the same for train as for titanic
- Make sure the distribution of Survived is the same for test as for titanic

In [8]:
train['target'].value_counts(normalize = True) * 100

target
1    51.324965
0    48.675035
Name: proportion, dtype: float64

In [9]:
test['target'].value_counts(normalize = True) * 100

target
1    51.298701
0    48.701299
Name: proportion, dtype: float64

# Q1 - Separate features and target
- Features from train data set in variable X
- Target from train data set in variable y
- Features from test data set in variable X_test
- Target from test data set in variable y_test

In [10]:
X = train.drop("target", axis = 1)
y = train['target']

X_test = test.drop('target', axis = 1)
y_test = test['target']

# Q2 - Create the numeric pipeline
- assign the numeric columns store in n_features 
- create the numeric pipeline called n_transformer
- 'imputer' use SimpleImputer strategy 'median'
- 'scaler' use StandardScaler

In [11]:
n_features = X.select_dtypes(include = np.number).columns
n_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median')),  
    ('scaler', StandardScaler())])   

# Q3 - Create a pipeline for processing Categorical data
- Assign categorical column names to cat_features
- Assign cat_transformer Pipeline
- Imputer missing values with name 'imputer' with SimpleImputer strategy = 'most_frequent'
- OneHotEncoder with name 'onehot' with handle_unknown = 'ignore'

In [12]:
cat_features = X.select_dtypes(include = ['object']).columns
cat_transformer = Pipeline([
     ('imputer', SimpleImputer(strategy = 'most_frequent')),
     ('onehot', OneHotEncoder(handle_unknown = 'ignore'))])

# Q4 - Creating a data preprocessing pipeline
- Assign preprocessor to ColumnTransformer
- n_transformer with name 'num' and the column names num_features
- cat_transformer with name 'cat' and the column names cat_features

In [13]:
preprocessor = ColumnTransformer([
    ('num', n_transformer, n_features),
    ('cat', cat_transformer, cat_features)])

# Q5 - Create the Decision Tree Model Pipeline
- from sklearn.tree import DecisionTreeClassifier
- Create a pipeline called decisionTree_pipeline
- 'preprocessor' and preprocessor
- 'classifier' and DecisionTreeClassifier with max_depth = 3, random_state = 42

In [14]:
from sklearn.tree import DecisionTreeClassifier

decisionTree_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=3, random_state=42))])

# Q6 - Cross Validation
- Using cross valuation called crossval_scores from the c_val_scores function with parameters decisionTree_pipeline, X, y, and cv = 5.
- print f"Average Cross-Validation Accuracy - train: {crossval_scores.mean():.2%"
- Using cross valuation called cv_scores1 from the cross_val_scores function with parameters decisionTree_pipeline, X_test, y_test, and cv = 5.
- print f"Average Cross-Validation Accuracy - test: {crossval_scores1.mean():.2%"

In [15]:
crossval_scores = cross_val_score(decisionTree_pipeline, X, y, cv = 5)
print(f'Average Cross-Validation Accuracy - train: {crossval_scores.mean():.2%}')

crossval_scores1 = cross_val_score(decisionTree_pipeline, X_test, y_test, cv = 5)
print(f"Average Cross-Validation Accuracy - test: {crossval_scores1.mean():.2%}")


Average Cross-Validation Accuracy - train: 83.26%
Average Cross-Validation Accuracy - test: 80.22%


# Q7 - Fit the Model & Predict
- fit the decisionTree_pipeline model for X and y
- decisionTree_pipeline method score for X_test, y_test assign it to test_acc
- print format f"Final Test (Holdout) Accuracy: {test_accuracy:.2%}"
- predict the y_pred from the dt_pipeline.predict method X
- predict the y_pred_test from the dt_pipeline.predict method X_test

In [16]:
decisionTree_pipeline.fit(X, y)
test_acc = decisionTree_pipeline.score(X_test, y_test)
print(f"Final Test (Holdout) Accuracy: {test_acc:.2%}")

Final Test (Holdout) Accuracy: 84.74%


In [17]:
# Predict from the model
y_pd = decisionTree_pipeline.predict(X)
y_pd_test = decisionTree_pipeline.predict(X_test)

# Q8 -  Create Model Evaluations
- from sklearn.metrics import classification_report
- from sklearn.metrics import ConfusionMatrixDisplay
- print on next line "\nClassification Report-Train:"
- print classification_report for train data y, y_pd
- print on next line "\nClassification Report-Test:"
- print classification_report for test data y_test, y_pd_test

In [18]:
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay

print("\nClassification Report-Train:")
print(classification_report(y, y_pd))
print("\nClassification Report-Test:")
print(classification_report(y_test, y_pd_test))


Classification Report-Train:
              precision    recall  f1-score   support

           0       0.82      0.89      0.85       349
           1       0.88      0.82      0.85       368

    accuracy                           0.85       717
   macro avg       0.85      0.85      0.85       717
weighted avg       0.85      0.85      0.85       717


Classification Report-Test:
              precision    recall  f1-score   support

           0       0.83      0.86      0.85       150
           1       0.86      0.84      0.85       158

    accuracy                           0.85       308
   macro avg       0.85      0.85      0.85       308
weighted avg       0.85      0.85      0.85       308



# Q9 - Random Forest Classification Model
- from sklearn.ensemble import RandomForestClassifier
- Create a random forest pipeline called randomforest
-   ('preprocessor', preprocessor),
-   ('rf', RandomForestClassifier(n_estimators = 100, random_state = 42))])
- Run cross validation called randomforest_scores = cross_val_score(randomforest_pipeline, X, y, cv = 5, scoring = 'accuracy')
- print(f"Accuracy per fold: {randomforest_scores}")
- print(f"Mean Accuracy: {randomforest_scores.mean(): .2%}")
- print(f"Standard Deviation: {randomforest_scores.std(): .4f}")
- fit the randomforest_pipeline for training variables
- predict training values assign to y_pd_rf_train.
- predict testing values assign to y_pd_rf_test.
- print on next line "\nRandom Forest Classification Report-Train:"
- print classification_report for train data y, y_pd_rf_train
- print on next line "\nRandom Forest Classification Report-Test:"
- print classification_report for test data y_test, y_pd_rf_test

In [19]:
from sklearn.ensemble import RandomForestClassifier
randomforest_pipeline = Pipeline([
    ('preprocessor', preprocessor), 
    ('rf', RandomForestClassifier(n_estimators = 100, random_state = 42))])
randomforest_scores = cross_val_score(randomforest_pipeline, X, y, cv = 5, scoring = 'accuracy')
print(f"Random Forest - Accuracy per fold: {randomforest_scores}")
print(f"Mean Accuracy: {randomforest_scores.mean(): .2%}")
print(f"Standard Deviation: {randomforest_scores.std(): .4f}")

Random Forest - Accuracy per fold: [0.95833333 0.95138889 0.97902098 0.95804196 0.97902098]
Mean Accuracy:  96.52%
Standard Deviation:  0.0116


In [20]:
randomforest_pipeline.fit(X, y)
y_pd_rf_train = randomforest_pipeline.predict(X)
y_pd_rf_test = randomforest_pipeline.predict(X_test)
print("\nRandom Forest - Classification Report-Train:")
print(classification_report(y, y_pd_rf_train))
print("\nRandom Forest - Classification Report-Test:")
print(classification_report(y_test, y_pd_rf_test))


Random Forest - Classification Report-Train:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       349
           1       1.00      1.00      1.00       368

    accuracy                           1.00       717
   macro avg       1.00      1.00      1.00       717
weighted avg       1.00      1.00      1.00       717


Random Forest - Classification Report-Test:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       150
           1       1.00      0.96      0.98       158

    accuracy                           0.98       308
   macro avg       0.98      0.98      0.98       308
weighted avg       0.98      0.98      0.98       308



# Q10 - Decision Tree - Extra Trees
- from sklearn.ensemble import ExtraTreesClassifier
- Create a extra trees pipeline called extraTree_pipeline 
-   ('preprocessor', preprocessor),
-   ('et', ExtraTreesClassifier(n_estimators = 100, random_state = 42))])
- Run cross validation called extraTree_scores = cross_val_score(extraTree_pipeline, X, y, cv = 5, scoring = 'accuracy')
- print(f"Extra Trees - Accuracy per fold: {extraTree_scores}")
- print(f"Mean Accuracy: {extraTree_scores.mean(): .2%}")
- print(f"Standard Deviation: {extraTree_scores.std(): .4f}")
- fit the extraTree_pipeline for training variables
- predict training values assign to y_pd_et_train.
- predict testing values assign to y_pd_et_test.
- print on next line "\nExtra Tree Classification Report-Train:"
- print classification_report for train data y, y_pd_et_train
- print on next line "\nExtra Tree Classification Report-Test:"
- print classification_report for test data y_test, y_pd_et_test

In [21]:
from sklearn.ensemble import ExtraTreesClassifier
extraTree_pipeline = Pipeline([
    ('preprocessor', preprocessor), 
    ('et', ExtraTreesClassifier(n_estimators = 100, random_state = 42))])
extraTree_scores = cross_val_score(extraTree_pipeline, X, y, cv = 5, scoring = 'accuracy')
print(f"Extra Trees Accuracy per fold: {extraTree_scores}")
print(f"Mean Accuracy: {extraTree_scores.mean(): .2%}")
print(f"Standard Deviation: {extraTree_scores.std(): .4f}")

Extra Trees Accuracy per fold: [0.95833333 0.97222222 0.97202797 0.99300699 0.97902098]
Mean Accuracy:  97.49%
Standard Deviation:  0.0113


In [22]:
extraTree_pipeline.fit(X, y)
y_pd_et_train = extraTree_pipeline.predict(X)
y_pd_et_test = extraTree_pipeline.predict(X_test)
print("\nExtra Trees - Classification Report-Train:")
print(classification_report(y, y_pd_et_train))
print("\nExtra Trees - Classification Report-Test:")
print(classification_report(y_test, y_pd_et_test))


Extra Trees - Classification Report-Train:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       349
           1       1.00      1.00      1.00       368

    accuracy                           1.00       717
   macro avg       1.00      1.00      1.00       717
weighted avg       1.00      1.00      1.00       717


Extra Trees - Classification Report-Test:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       150
           1       1.00      0.98      0.99       158

    accuracy                           0.99       308
   macro avg       0.99      0.99      0.99       308
weighted avg       0.99      0.99      0.99       308

